# Grocery Sales Forecasting - EDA and Feature Engineering
This notebook demonstrates the exploratory data analysis and feature engineering pipeline for the supermarket sales forecasting model. It is designed to be fully reproducible, modular, and explains the rationale behind each feature and validation split.

## 1. Imports and Configurations

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from features import build_features

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Load Raw Tables
The dataset consists of several relational tables:
- `train.csv` / `test.csv`: Core sales transactions and promotions
- `stores.csv`: Store location and type metadata
- `oil.csv`: Daily oil prices (economic indicator)
- `holidays_events.csv`: Calendar of local, regional, and national holidays

In [ ]:
data_dir = '../data'
train_raw = pd.read_csv(os.path.join(data_dir, 'train.csv'))
stores_raw = pd.read_csv(os.path.join(data_dir, 'stores.csv'))
oil_raw = pd.read_csv(os.path.join(data_dir, 'oil.csv'))
holidays_raw = pd.read_csv(os.path.join(data_dir, 'holidays_events.csv'))

print(f"Train size: {train_raw.shape}")
print(f"Stores size: {stores_raw.shape}")
print(f"Oil size: {oil_raw.shape}")
print(f"Holidays size: {holidays_raw.shape}")

## 3. Exploratory Data Analysis

### 3.1 Target Distribution
Grocery sales figures are heavily right-skewed. To stabilize variance and optimize for RMSLE directly, we apply a log-transform: $\log(1 + x)$.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(train_raw['sales'], bins=50, kde=True, ax=ax[0])
ax[0].set_title("Raw Sales Distribution")

sns.histplot(np.log1p(train_raw['sales']), bins=50, kde=True, ax=ax[1])
ax[1].set_title("Log-transformed Sales")
plt.show()

### 3.2 Oil Prices & Linear Interpolation
Oil prices are missing on weekends. We apply linear interpolation to represent a continuous economic index.

In [ ]:
oil_raw['date'] = pd.to_datetime(oil_raw['date'])
full_dates = pd.date_range(start=oil_raw['date'].min(), end=oil_raw['date'].max())
oil_filled = oil_raw.set_index('date').reindex(full_dates).interpolate(method='linear', limit_direction='both').reset_index()

plt.plot(oil_raw['date'], oil_raw['dcoilwtico'], label='Raw (With Gaps)', alpha=0.6)
plt.plot(oil_filled['index'], oil_filled['dcoilwtico'], label='Interpolated', alpha=0.4, linestyle='--')
plt.title("Ecuador Daily Oil Price Trend")
plt.legend()
plt.show()

## 4. Run Feature Pipeline

In [ ]:
train_feat, test_feat = build_features(
    os.path.join(data_dir, 'train.csv'),
    os.path.join(data_dir, 'test.csv'),
    os.path.join(data_dir, 'stores.csv'),
    os.path.join(data_dir, 'oil.csv'),
    os.path.join(data_dir, 'holidays_events.csv')
)

print(f"Processed Train dimensions: {train_feat.shape}")
print(f"Processed Test dimensions: {test_feat.shape}")

## 5. Review Lag Features and Rolling Statistics
To avoid data leakage during the 16-day forecasting window, the minimum lag shift is set to 16 days. Below we verify that the lags correspond to past sales data only.

In [ ]:
sample_cols = ['date', 'store_nbr', 'family', 'sales', 'sales_lag_16', 'sales_roll_mean_7']
print(train_feat[sample_cols].head(20))